# Final Paper Plots: Representative Topics

> Validation note: canonical rescue results now come from the SCCKN Gemma 4 31B pipeline in `notebooks/llm_validation/` (8,558 final rescues). Gabriel outputs are historical.


In [ ]:
from pathlib import Path
import sys

def find_project_root() -> Path:
    """Anchor on config.txt — safe from any subdir depth."""
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "config.txt").exists():
            return p
    raise RuntimeError("config.txt not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text as sk_text

from src.io import get_figure_dir, get_gabriel_dir, get_interim_dir, get_processed_dir, get_test_mode, get_topic_dir, load_table
from src.topics import (
    _EXTRA_STOP_WORDS as _TOPIC_EXTRA_STOPS,
    add_topic_display_labels,
    get_cluster_approval_columns,
)

# Mirror stop_tokens from shorten_topic_name (not exported from topics.py)
_TOPIC_NAME_STOPS = {
    "http", "https", "html", "www", "com", "news", "story",
    "article", "read", "nnn", "cn", "que",
}

# Community Notes platform noise
_CN_NOISE = {
    "sns", "nhk", "post", "posts", "opinion", "note", "tweet", "tweets",
    "community", "notes", "context", "claim", "claims", "source",
    "sources", "video", "image", "photo", "says", "said", "stated",
    "states", "added", "wrote", "writing",
}

_ALL_STOP_WORDS = list(
    sk_text.ENGLISH_STOP_WORDS
    .union(_TOPIC_EXTRA_STOPS)
    .union(_TOPIC_NAME_STOPS)
    .union(_CN_NOISE)
)

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 1200)

TEST_MODE = get_test_mode()
print(f'=== TEST_MODE = {TEST_MODE}'
      f' (' + ('smoke test (small sample)' if TEST_MODE else 'FULL DATA') + ') ===')


def topic_label_column(df: pd.DataFrame) -> str:
    return 'topic_display_label' if 'topic_display_label' in df.columns else 'topic_label'


def avg_cluster_columns(df: pd.DataFrame) -> list[str]:
    return sorted(
        [col for col in df.columns if col.startswith('avg_cluster_')],
        key=lambda col: int(col.rsplit('_', 1)[1]),
    )


def cluster_id_from_avg_col(col: str) -> int:
    return int(col.rsplit('_', 1)[1])


def ordered_unique(values) -> list:
    return list(dict.fromkeys(list(values)))


In [ ]:
INTERIM_DIR = get_interim_dir()
PROCESSED_DIR = get_processed_dir()
TOPIC_DIR = get_topic_dir()
GABRIEL_CACHE_DIR = get_gabriel_dir() / 'cache'
FIG_DIR = get_figure_dir()
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Colour palette ────────────────────────────────────────────────────────────
# Core design principles:
#   • Wong (2011) Nature Methods colorblind-safe base
#   • Blue family → trust, reliability, intellectual depth
#   • Amber/gold  → curiosity, warmth, importance
#   • Teal green  → positive outcome, balance
#   • Burnt red   → danger/troll (used sparingly)
# ─────────────────────────────────────────────────────────────────────────────
PALETTE = {
    # Structural
    'ink':   '#1B2A3B',   # Deep navy — text, axes, borders
    'slate': '#5B7FA6',   # Periwinkle-gray — secondary text, connector lines
    'fog':   '#CBDcED',   # Very light blue — backgrounds, lollipop connectors

    # Cluster identity (Wong 2011, colorblind-safe)
    'cluster0': '#D55E00',   # Vermillion — warm, energetic, clearly distinct
    'cluster1': '#0072B2',   # Cerulean blue — cool, trustworthy, scientific
    'cluster2': '#009E73',   # Teal green — middle/mixed bloc in 3-cluster runs

    # Strategy colours
    'simple_majoritarian': '#56B4E9',  # Sky blue — clear, approachable
    'representative':      '#E69F00',  # Golden amber — important, stands out

    # Status category colours (Helpful / NMR / Other)
    'success': '#009E73',   # Teal green  — Helpful (positive outcome)
    'sand':    '#F2C97E',   # Warm peach  — NMR (neutral, clearly visible on white)
    'other':   '#9BB8D3',   # Soft steel blue — Other (replaces slate-gray; text readable)

    # Accent & special
    'accent': '#CC79A7',   # Muted mauve — accent, miscellaneous (Wong reddish-purple)
    'danger': '#B5351A',   # Deep burnt red — troll/danger (used sparingly)
}

sns.set_theme(
    style='whitegrid',
    context='paper',
    rc={
        'figure.dpi': 160,
        'savefig.dpi': 320,
        'axes.facecolor': 'white',
        'figure.facecolor': 'white',
        'axes.edgecolor': '#BFD0E0',
        'grid.color': '#DDE8F0',
        'grid.linewidth': 0.8,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.titleweight': 'bold',
        'axes.labelweight': 'bold',
        'font.size': 10,
        'axes.titlesize': 12,
        'axes.labelsize': 10,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'legend.fontsize': 9
    }
)

def save_figure(fig, stem: str):
    fig.savefig(FIG_DIR / f'{stem}.pdf', bbox_inches='tight', facecolor='white')
    fig.savefig(FIG_DIR / f'{stem}.png', bbox_inches='tight', facecolor='white')
    print(f'Saved {stem}.pdf and {stem}.png')

def save_unavailable_figure(stem: str, title: str, message: str):
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    ax.axis('off')
    ax.text(0.5, 0.58, title, ha='center', va='center', fontsize=13, fontweight='bold', color=PALETTE['ink'])
    ax.text(0.5, 0.43, message, ha='center', va='center', fontsize=10, color=PALETTE['slate'], wrap=True)
    fig.tight_layout()
    save_figure(fig, stem)

def clean_gabriel_label(x):
    if x is None:
        return 'missing'
    if isinstance(x, float) and pd.isna(x):
        return 'missing'
    if isinstance(x, (list, tuple)):
        return str(x[0]) if len(x) else 'missing'
    if hasattr(x, 'tolist') and not isinstance(x, str):
        x = x.tolist()
        if isinstance(x, list):
            return str(x[0]) if len(x) else 'missing'
    s = str(x).strip()
    if s in {'', '[]', '[ ]'}:
        return 'missing'
    if s.startswith('[') and s.endswith(']'):
        s = s[1:-1].strip().strip("'").strip('"')
    return s or 'missing'


BASE_CLUSTER_COLORS = ['#D55E00', '#0072B2', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']

def color_for_cluster(cluster_id: int) -> str:
    return BASE_CLUSTER_COLORS[int(cluster_id) % len(BASE_CLUSTER_COLORS)]


In [ ]:
cluster_summary = load_table(INTERIM_DIR / 'cluster_summary.parquet')
silhouette_over_k = load_table(INTERIM_DIR / 'silhouette_over_k.parquet')
stability_over_k = load_table(INTERIM_DIR / 'stability_over_k.parquet')
# Load only the columns needed for Figure A3 to stay within the memory budget.
_rc_slim = pd.read_parquet(
    INTERIM_DIR / 'ratings_clustered.parquet',
    columns=['raterParticipantId', 'cluster', 'vote'],
)
_mb = pd.read_parquet(
    INTERIM_DIR / 'user_clusters_method_b_voteprofile.parquet',
    columns=['raterParticipantId', 'cluster'],
)
_rc_slim = _rc_slim.rename(columns={'cluster': 'initial_cluster'})
ratings_clustered = _rc_slim.merge(
    _mb,
    on='raterParticipantId',
    how='left',
    validate='many_to_one',
)
del _rc_slim

scores = load_table(PROCESSED_DIR / 'scores.parquet')
rescue_summary = load_table(PROCESSED_DIR / 'rescue_summary.parquet')
selection_log = load_table(PROCESSED_DIR / 'selection_log.parquet')
selection_status_summary = load_table(PROCESSED_DIR / 'selection_status_summary.parquet')
topic_notes = load_table(TOPIC_DIR / 'topic_notes.parquet')
topic_cluster_stats = load_table(TOPIC_DIR / 'topic_cluster_stats.parquet')
topic_strategy_summary = load_table(TOPIC_DIR / 'topic_strategy_summary.parquet')

gabriel_cache_paths = {
    'politics': GABRIEL_CACHE_DIR / 'politics_classification_cache.parquet',
    'troll': GABRIEL_CACHE_DIR / 'troll_classification_cache.parquet',
    'rating': GABRIEL_CACHE_DIR / 'rescue_rating_cache.parquet',
}
GABRIEL_AVAILABLE = all(path.exists() for path in gabriel_cache_paths.values())
if GABRIEL_AVAILABLE:
    politics_cache = load_table(gabriel_cache_paths['politics'])
    troll_cache = load_table(gabriel_cache_paths['troll'])
    rating_cache = load_table(gabriel_cache_paths['rating'])
else:
    missing = [str(path) for path in gabriel_cache_paths.values() if not path.exists()]
    print('Gabriel cache files missing; Gabriel figures will be marked unavailable:')
    for path in missing:
        print(f'  - {path}')
    politics_cache = pd.DataFrame(columns=['noteId', 'politics_predicted_classes'])
    troll_cache = pd.DataFrame(columns=['noteId', 'troll_predicted_classes'])
    rating_cache = pd.DataFrame(columns=['noteId', 'rescue_worthiness', 'informational_value', 'evidentiary_specificity', 'troll_likelihood', 'clarity'])


topic_notes = add_topic_display_labels(topic_notes)
topic_cluster_stats = add_topic_display_labels(topic_cluster_stats)
topic_strategy_summary = add_topic_display_labels(topic_strategy_summary)
TOPIC_LABEL_COL = topic_label_column(topic_cluster_stats)


In [ ]:
# Cluster role variables — always computed (needed by figures regardless of Gabriel)
cluster_role = cluster_summary[['cluster', 'users', 'avg_positive_rate']].copy()
cluster_role['cluster'] = cluster_role['cluster'].astype(int)
ordered_cluster_ids = cluster_role.sort_values('avg_positive_rate')['cluster'].tolist()

cluster_role_name = {}
cluster_role_short = {}
if len(ordered_cluster_ids) == 1:
    only = ordered_cluster_ids[0]
    cluster_role_name[only] = 'Only bloc'
    cluster_role_short[only] = 'only bloc'
else:
    cluster_role_name[ordered_cluster_ids[0]] = 'More skeptical bloc'
    cluster_role_short[ordered_cluster_ids[0]] = 'skeptical bloc'
    cluster_role_name[ordered_cluster_ids[-1]] = 'More approving bloc'
    cluster_role_short[ordered_cluster_ids[-1]] = 'approving bloc'
    middle = ordered_cluster_ids[1:-1]
    for pos, cluster_id in enumerate(middle, start=1):
        suffix = '' if len(middle) == 1 else f' {pos}'
        cluster_role_name[cluster_id] = f'Middle bloc{suffix}'
        cluster_role_short[cluster_id] = f'middle bloc{suffix}'

for cluster_id in cluster_role['cluster']:
    cluster_role_name.setdefault(cluster_id, f'Cluster {cluster_id}')
    cluster_role_short.setdefault(cluster_id, f'cluster {cluster_id}')

cluster_color = {cluster_id: color_for_cluster(cluster_id) for cluster_id in cluster_role['cluster']}

# Gabriel-specific: only run when cache files are present
if not GABRIEL_AVAILABLE:
    print("Skip: Gabriel data not available.")
    gabriel_merged = pd.DataFrame()
else:
    rep_rescued = selection_log[
        (selection_log['strategy'] == 'Representative')
        & (selection_log['status'] != 'CURRENTLY_RATED_HELPFUL')
        & (selection_log['passes_bridge_threshold'].fillna(False))
    ].copy()

    note_topic_cols = ['noteId', 'topic_label']
    if 'topic_display_label' in topic_notes.columns:
        note_topic_cols.append('topic_display_label')

    gabriel_merged = (
        rep_rescued[['tweetId', 'selected_noteId', 'status', 'global_approval', 'bridge_score']]
        .drop_duplicates(['tweetId', 'selected_noteId'])
        .merge(scores, left_on='selected_noteId', right_on='noteId', how='left', suffixes=('_selection', ''))
        .merge(topic_notes[note_topic_cols].drop_duplicates('noteId'), on='noteId', how='left', suffixes=('', '_topic'))
        .merge(politics_cache[['noteId', 'politics_predicted_classes']], on='noteId', how='left')
        .merge(troll_cache[['noteId', 'troll_predicted_classes']], on='noteId', how='left')
        .merge(rating_cache[['noteId', 'rescue_worthiness', 'informational_value', 'evidentiary_specificity', 'troll_likelihood', 'clarity']], on='noteId', how='left')
    )

    cluster_approval_cols = get_cluster_approval_columns(gabriel_merged)
    if cluster_approval_cols:
        gabriel_merged['dominant_cluster'] = (
            gabriel_merged[cluster_approval_cols]
            .astype(float)
            .idxmax(axis=1)
            .str.extract(r'cluster_(\d+)_approval')[0]
        )
        gabriel_merged['dominant_cluster'] = gabriel_merged['dominant_cluster'].astype('Int64')
    else:
        gabriel_merged['dominant_cluster'] = pd.Series(pd.NA, index=gabriel_merged.index, dtype='Int64')
    gabriel_merged['dominant_cluster_label'] = gabriel_merged['dominant_cluster'].map(cluster_role_name).fillna('Unknown bloc')
    gabriel_merged['politics_clean'] = gabriel_merged['politics_predicted_classes'].apply(clean_gabriel_label)
    gabriel_merged['troll_clean'] = gabriel_merged['troll_predicted_classes'].apply(clean_gabriel_label)


## Figure 1: Cluster Diagnostics

In [ ]:
diag = silhouette_over_k.merge(stability_over_k[['k', 'mean_ari']], on='k', how='left').sort_values('k')
chosen_k = int(cluster_summary['cluster'].nunique())
fig, ax1 = plt.subplots(figsize=(9.5, 5.5))
ax2 = ax1.twinx()
ax1.plot(diag['k'], diag['silhouette'], marker='o', markersize=6, linewidth=2.5, color=PALETTE['simple_majoritarian'])
ax2.plot(diag['k'], diag['mean_ari'], marker='s', markersize=5.5, linewidth=2.2, color=PALETTE['representative'])
if chosen_k in set(diag['k']):
    chosen_silhouette = diag.loc[diag['k'] == chosen_k, 'silhouette'].iloc[0]
    ax1.axvline(chosen_k, color=PALETTE['ink'], linestyle='--', linewidth=1.3, alpha=0.8)
    ax1.annotate(
        f'Chosen $K={chosen_k}$',
        xy=(chosen_k, chosen_silhouette),
        xytext=(chosen_k + 0.35, chosen_silhouette),
        fontsize=9,
        color=PALETTE['ink'],
    )
ax1.set_title(f'Cluster Diagnostics for the Selected {chosen_k}-Bloc Structure', pad=12)
ax1.set_xlabel('Number of clusters $K$')
ax1.set_ylabel('Silhouette score', color=PALETTE['simple_majoritarian'])
ax2.set_ylabel('Mean adjusted Rand index', color=PALETTE['representative'])
ax1.tick_params(axis='y', colors=PALETTE['simple_majoritarian'])
ax2.tick_params(axis='y', colors=PALETTE['representative'])
handles = [
    Line2D([0], [0], color=PALETTE['simple_majoritarian'], marker='o', lw=2.5, label='Silhouette'),
    Line2D([0], [0], color=PALETTE['representative'], marker='s', lw=2.2, label='Mean ARI')
]
ax1.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.12), ncol=2, frameon=False)
fig.tight_layout()
save_figure(fig, 'figure_01_cluster_diagnostics')


## Figure 3: Exact Status Counts by Strategy

In [ ]:
status_map = {
    'CURRENTLY_RATED_HELPFUL': 'Helpful',
    'NEEDS_MORE_RATINGS': 'NMR',
    'NEEDS_MORE_RATINGS_WITH_OLD_TWEET': 'NMR'
}

simple_majoritarian_pool = selection_log[
    (selection_log['strategy'] == 'Simple Majoritarian Rule')
    & (selection_log['tweetId'].isin(rescue_summary.loc[rescue_summary['simple_majoritarian_published'], 'tweetId']))
].copy()
simple_majoritarian_pool['strategy_label'] = 'Simple Majoritarian Rule'

rep_pool = selection_log[
    (selection_log['strategy'] == 'Representative')
    & (selection_log['tweetId'].isin(rescue_summary.loc[rescue_summary['rep_rescued'], 'tweetId']))
].copy()
rep_pool['strategy_label'] = 'Representative'

pluralistic_pools = []
for cluster_id in ordered_cluster_ids:
    pool = selection_log[
        (selection_log['strategy'] == f'Cluster {cluster_id}')
        & (selection_log['status'] != 'CURRENTLY_RATED_HELPFUL')
    ].copy()
    pool['strategy_label'] = f"Pluralistic ({cluster_role_short[cluster_id]})"
    pluralistic_pools.append(pool)

rescue_pool = pd.concat([simple_majoritarian_pool, rep_pool] + pluralistic_pools, ignore_index=True)
rescue_pool['status_group'] = rescue_pool['status'].map(status_map).fillna('Other')
rescue_pool['status_group'] = pd.Categorical(rescue_pool['status_group'], categories=['Helpful', 'NMR', 'Other'], ordered=True)
strategy_order = ['Simple Majoritarian Rule', 'Representative'] + [f"Pluralistic ({cluster_role_short[cid]})" for cid in ordered_cluster_ids]
pivot = rescue_pool.groupby(['strategy_label', 'status_group'])['selected_noteId'].nunique().unstack(fill_value=0).reindex(strategy_order).fillna(0)

fig, ax = plt.subplots(figsize=(max(9.2, 1.6 * len(pivot)), 6.5))
bottom = np.zeros(len(pivot))
status_colors = {'Helpful': PALETTE['success'], 'NMR': PALETTE['sand'], 'Other': PALETTE['other']}
for status in ['Helpful', 'NMR', 'Other']:
    vals = pivot[status].values if status in pivot.columns else np.zeros(len(pivot))
    ax.bar(pivot.index, vals, bottom=bottom, color=status_colors[status], edgecolor='white', linewidth=0.7, label=status)
    for i, (b, v) in enumerate(zip(bottom, vals)):
        if v > 0:
            ax.text(i, b + v / 2, f'{int(v):,}', ha='center', va='center', fontsize=9.5, fontweight='semibold')
    bottom = bottom + vals
ax.set_title('NMR Rescue Counts by Decision Rule and Voting Bloc')
ax.set_xlabel('')
ax.set_ylabel('Selected note count')
ax.set_xticklabels(pivot.index, rotation=25, ha='right')
ax.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.12))
fig.tight_layout()
save_figure(fig, 'figure_03_strategy_status_counts')


## Figure 4: High-Disagreement Topics with Representative Pick Count

In [ ]:
rep_topic = topic_strategy_summary[topic_strategy_summary['strategy'] == 'Representative'].copy()
label_col = topic_label_column(rep_topic)

rep_topic_counts = rep_topic.groupby(['topic', label_col, 'status_group'], as_index=False)['selected_picks'].sum()
rep_topic_counts = rep_topic_counts.rename(columns={label_col: 'topic_plot_label'})
rep_topic_totals = rep_topic_counts.groupby(['topic', 'topic_plot_label'], as_index=False)['selected_picks'].sum().rename(columns={'selected_picks': 'rep_total'})

topic_plot = topic_cluster_stats[['topic', TOPIC_LABEL_COL, 'abs_gap', 'notes']].rename(columns={TOPIC_LABEL_COL: 'topic_plot_label'}).merge(rep_topic_totals, on=['topic', 'topic_plot_label'], how='inner')
topic_plot = topic_plot[(topic_plot['rep_total'] > 0) & (topic_plot['notes'] >= 5)].sort_values(['abs_gap', 'rep_total'], ascending=[False, False]).head(10)
topic_order = ordered_unique(topic_plot.sort_values('abs_gap')['topic_plot_label'])
topic_plot = topic_plot.set_index('topic_plot_label').loc[topic_order].reset_index()

rep_topic_detail = rep_topic_counts[rep_topic_counts['topic_plot_label'].isin(topic_order)].copy()
rep_topic_detail['topic_plot_label'] = pd.Categorical(rep_topic_detail['topic_plot_label'], categories=topic_order, ordered=True)
rep_topic_detail = rep_topic_detail.sort_values('topic_plot_label')

fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.6), gridspec_kw={'width_ratios': [0.95, 1.25]})
axes[0].hlines(y=topic_plot['topic_plot_label'], xmin=0, xmax=topic_plot['abs_gap'], color=PALETTE['fog'], linewidth=2.4)
axes[0].scatter(topic_plot['abs_gap'], topic_plot['topic_plot_label'], s=np.clip(topic_plot['rep_total'] * 12, 55, 260), color=PALETTE['representative'], edgecolor='white', linewidth=0.6)
axes[0].set_title('High-disagreement topic ranking')
axes[0].set_xlabel('Approval range across clusters')
axes[0].set_ylabel('Topic label')
axes[0].tick_params(axis='y', labelsize=10.5)

status_colors = {'Helpful': PALETTE['success'], 'NMR': PALETTE['sand'], 'Other': PALETTE['other']}
pivot = rep_topic_detail.pivot(index='topic_plot_label', columns='status_group', values='selected_picks').fillna(0).loc[topic_order]
left = np.zeros(len(pivot))
for status in ['Helpful', 'NMR', 'Other']:
    vals = pivot[status].values if status in pivot.columns else np.zeros(len(pivot))
    axes[1].barh(pivot.index, vals, left=left, color=status_colors[status], edgecolor='white', linewidth=0.6, label=status)
    left = left + vals
axes[1].set_title('Representative pick count by status')
axes[1].set_xlabel('Selected picks')
axes[1].set_ylabel('')
axes[1].tick_params(axis='y', labelsize=10.5)
axes[1].legend(frameon=False, loc='upper right')

fig.suptitle('Representative Selection Still Concentrates in the Most Contentious Topic Areas', y=1.02, fontsize=13, fontweight='bold')
fig.tight_layout()
save_figure(fig, 'figure_04_high_disagreement_topics')


## Figure 5: Cluster Positivity in High-Disagreement Topics

In [ ]:
avg_cols = avg_cluster_columns(topic_cluster_stats)
cluster_polarity = topic_cluster_stats[[TOPIC_LABEL_COL, 'abs_gap', 'notes'] + avg_cols].copy()
cluster_polarity = cluster_polarity[(cluster_polarity['notes'] >= 5)].sort_values('abs_gap', ascending=False).head(10)
cluster_polarity = cluster_polarity.sort_values('abs_gap')

fig, ax = plt.subplots(figsize=(10.2, 6.1))
for _, row in cluster_polarity.iterrows():
    vals = row[avg_cols].astype(float)
    y = row[TOPIC_LABEL_COL]
    ax.plot([vals.min(), vals.max()], [y, y], color=PALETTE['fog'], linewidth=2.3, zorder=1)
    for col in avg_cols:
        cluster_id = cluster_id_from_avg_col(col)
        ax.scatter(row[col], y, s=80, color=cluster_color.get(cluster_id, color_for_cluster(cluster_id)), edgecolor='white', linewidth=0.6, label=cluster_role_name.get(cluster_id, f'Cluster {cluster_id}'), zorder=3)

# Deduplicate legend entries
handles, labels = ax.get_legend_handles_labels()
legend = dict(zip(labels, handles))
ax.set_xlim(-0.02, 1.02)
ax.set_xlabel('Average approval within topic')
ax.set_ylabel('Topic label')
ax.set_title('Which Voting Bloc Is More Positive in the Most Polarized Topics?')
ax.legend(legend.values(), legend.keys(), frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.16), ncol=min(3, len(legend)))
fig.tight_layout()
save_figure(fig, 'figure_05_topic_cluster_polarity')


## Figure 6: Gabriel Political Classification

In [ ]:
if GABRIEL_AVAILABLE:
    politics_counts = gabriel_merged['politics_clean'].value_counts().rename_axis('label').reset_index(name='count')
    politics_counts = politics_counts[politics_counts['label'] != 'missing'].copy()
    
    if politics_counts.empty:
        save_unavailable_figure(
            'figure_06_gabriel_politics',
            'Gabriel Political Classification Unavailable',
            'The SCC plotting job did not find Gabriel classification caches. Run the Gabriel notebook locally after syncing processed outputs to generate this figure.'
        )
    else:
        politics_counts['share'] = politics_counts['count'] / politics_counts['count'].sum()
        politics_counts = politics_counts.sort_values('count')
    
        fig, ax = plt.subplots(figsize=(8.4, 4.9))
        # Gradient from warm amber (representative) to deep cerulean (cluster1) — palette-consistent
        bars = ax.barh(
            politics_counts['label'], politics_counts['count'],
            color=sns.color_palette(f"blend:{PALETTE['sand']},{PALETTE['cluster1']}", n_colors=len(politics_counts)),
            edgecolor='white', linewidth=0.6
        )
        ax.set_title('Political Classification of Representative-Rescued Notes')
        ax.set_xlabel('Rescued notes')
        ax.set_ylabel('Gabriel political class')
        for bar, (_, row) in zip(bars, politics_counts.iterrows()):
            ax.text(bar.get_width() + 1.3, bar.get_y() + bar.get_height() / 2,
                    f"{row['count']:,} ({row['share']:.0%})", va='center', fontsize=8.8)
        ax.text(0.98, 0.05, 'Color gradient distinguishes categories only;\nbar length encodes rescued-note counts.',
                transform=ax.transAxes, ha='right', va='bottom', fontsize=8.2, color=PALETTE['slate'],
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=PALETTE['fog']))
        fig.tight_layout()
        save_figure(fig, 'figure_06_gabriel_politics')
    
else:
    print(f"Skip Cell 14: Gabriel data not available.")

## Figure 7: Gabriel Rating Profile

In [ ]:
if GABRIEL_AVAILABLE:
    rating_cols = {
        'rescue_worthiness': 'Rescue worthiness',
        'informational_value': 'Informational value',
        'evidentiary_specificity': 'Evidentiary specificity',
        'clarity': 'Clarity',
        'troll_likelihood': 'Troll likelihood'
    }
    rating_summary = []
    for col, label in rating_cols.items():
        series = gabriel_merged[col].dropna()
        if series.empty:
            continue
        mean = series.mean()
        sem = series.std(ddof=1) / np.sqrt(len(series)) if len(series) > 1 else 0
        rating_summary.append({'metric': label, 'mean': mean, 'low': mean - 1.96 * sem, 'high': mean + 1.96 * sem})
    rating_summary = pd.DataFrame(rating_summary)
    
    if rating_summary.empty:
        save_unavailable_figure(
            'figure_07_gabriel_ratings',
            'Gabriel Rating Profile Unavailable',
            'The SCC plotting job did not find Gabriel rating caches. Run the Gabriel notebook locally after syncing processed outputs to generate this figure.'
        )
    else:
        rating_summary = rating_summary.sort_values('mean')
        fig, ax = plt.subplots(figsize=(8.5, 4.8))
        metric_colors = {
            'Rescue worthiness': PALETTE['representative'],
            'Informational value': PALETTE['simple_majoritarian'],
            'Evidentiary specificity': PALETTE['accent'],
            'Clarity': PALETTE['success'],
            'Troll likelihood': PALETTE['danger']
        }
        ax.hlines(y=rating_summary['metric'], xmin=rating_summary['low'], xmax=rating_summary['high'], color=PALETTE['slate'], linewidth=2.0)
        ax.scatter(rating_summary['mean'], rating_summary['metric'], s=85, color=rating_summary['metric'].map(metric_colors))
        ax.axvline(50, color=PALETTE['fog'], linestyle='--', linewidth=1.0)
        ax.set_xlim(0, 100)
        ax.set_xlabel('Average Gabriel score')
        ax.set_ylabel('')
        ax.set_title('Representative-Rescued Notes Score High on Clarity and Moderately High on Worthiness')
        for _, row in rating_summary.iterrows():
            ax.text(row['high'] + 1.0, row['metric'], f"{row['mean']:.1f}", va='center', fontsize=8.8)
        fig.tight_layout()
        save_figure(fig, 'figure_07_gabriel_ratings')
    
else:
    print(f"Skip Cell 16: Gabriel data not available.")

## Figure 8: Troll-Labelled Notes by Topic Pocket

In [ ]:
if GABRIEL_AVAILABLE:
    troll_subset = gabriel_merged[gabriel_merged['troll_clean'] == 'likely_troll_or_bad_faith'].copy()
    cluster_counts = troll_subset['dominant_cluster_label'].value_counts().rename_axis('dominant_cluster').reset_index(name='count')
    topic_counts = troll_subset[topic_label_column(troll_subset)].fillna('Unknown topic').value_counts().head(10).rename_axis('topic_label').reset_index(name='count').sort_values('count')
    
    if cluster_counts.empty or topic_counts.empty:
        save_unavailable_figure(
            'figure_08_troll_topic_pockets',
            'Gabriel Troll-Pocket Analysis Unavailable',
            'The SCC plotting job did not find Gabriel troll-classification caches or no troll-labelled notes were present in this smoke-test slice.'
        )
    else:
        fig, axes = plt.subplots(1, 2, figsize=(11.8, 6.5), gridspec_kw={'width_ratios': [0.9, 1.35]})
        sns.barplot(data=cluster_counts, x='dominant_cluster', y='count', hue='dominant_cluster', dodge=False, palette={cluster_role_name[cid]: cluster_color[cid] for cid in cluster_color}, legend=False, ax=axes[0])
        axes[0].set_title('By dominant cluster')
        axes[0].set_xlabel('')
        axes[0].set_ylabel('Troll-labelled notes')
        for idx, row in cluster_counts.iterrows():
            axes[0].text(idx, row['count'] + 0.08, int(row['count']), ha='center', va='bottom', fontsize=9, fontweight='bold')
    
        sns.barplot(data=topic_counts, x='count', y='topic_label', color=PALETTE['danger'], ax=axes[1])
        axes[1].set_title('By topic pocket')
        axes[1].set_xlabel('Troll-labelled notes')
        axes[1].set_ylabel('')
    
        fig.suptitle('Troll-Labelled Failure Pockets Remain Small but Cluster in Specific Topic Areas', y=1.03, fontsize=13, fontweight='bold')
        fig.tight_layout()
        save_figure(fig, 'figure_08_troll_topic_pockets')
    
else:
    print(f"Skip Cell 18: Gabriel data not available.")

## Figure A1: Cluster-Based Note Text Analysis (TF-IDF)

In [ ]:
# Figure A1 — Cluster-Based Note Text Analysis (TF-IDF)
HIGH_APPROVAL_THRESHOLD = 0.8
TFIDF_TOP_N = 15
cluster_cols = get_cluster_approval_columns(topic_notes)
cluster_ids = [int(col.split('_')[1]) for col in cluster_cols]

corpus_a1 = []
labels_a1 = []
for cluster_id, col in zip(cluster_ids, cluster_cols):
    cluster_corpus = topic_notes[topic_notes[col] >= HIGH_APPROVAL_THRESHOLD]['summary'].dropna().astype(str).tolist()
    corpus_a1.extend(cluster_corpus)
    labels_a1.extend([cluster_id] * len(cluster_corpus))

if not corpus_a1 or len(set(labels_a1)) < 2:
    save_unavailable_figure(
        'figure_A1_cluster_tfidf',
        'Cluster TF-IDF Unavailable',
        'Not enough high-approval note text was available across clusters to compute differential TF-IDF.'
    )
else:
    vectorizer_a1 = TfidfVectorizer(
        max_features=8000,
        ngram_range=(1, 2),
        stop_words=_ALL_STOP_WORDS,
        min_df=3,
        max_df=0.85,
    )
    tfidf_mat_a1 = vectorizer_a1.fit_transform(corpus_a1)
    feat_names_a1 = vectorizer_a1.get_feature_names_out()

    mean_by_cluster = {}
    for cluster_id in cluster_ids:
        idx = [i for i, label in enumerate(labels_a1) if label == cluster_id]
        if idx:
            mean_by_cluster[cluster_id] = np.asarray(tfidf_mat_a1[idx].mean(axis=0)).ravel()

    n_clusters = len(mean_by_cluster)
    fig, axes = plt.subplots(1, n_clusters, figsize=(5.4 * n_clusters, 7.2), squeeze=False)
    axes = axes.ravel()
    for ax, cluster_id in zip(axes, mean_by_cluster):
        current = mean_by_cluster[cluster_id]
        others = [vec for cid, vec in mean_by_cluster.items() if cid != cluster_id]
        other_mean = np.vstack(others).mean(axis=0)
        diff = current - other_mean
        top_idx = np.argsort(diff)[::-1][:TFIDF_TOP_N]
        df_terms = pd.DataFrame({
            'term': feat_names_a1[top_idx],
            'differential': diff[top_idx],
        }).sort_values('differential')
        ax.barh(df_terms['term'], df_terms['differential'], color=cluster_color.get(cluster_id, color_for_cluster(cluster_id)), alpha=0.9, edgecolor='white', linewidth=0.4)
        ax.axvline(0, color=PALETTE['ink'], linewidth=0.6, linestyle='--')
        ax.set_title(f"{cluster_role_name.get(cluster_id, f'Cluster {cluster_id}')} — Distinctive Terms\n(vs. other blocs)", fontsize=11)
        ax.set_xlabel('Differential TF-IDF')
        ax.tick_params(axis='y', labelsize=10.5)

    fig.suptitle('What Content Does Each Voting Bloc Endorse? Bloc-Level TF-IDF Comparison', fontsize=13, fontweight='bold', y=1.02)
    fig.tight_layout()
    save_figure(fig, 'figure_A1_cluster_tfidf')


## Figure A2: Systematized Disagreement Direction

In [ ]:
# Figure A2a — Bar: top 50 topics by cross-cluster approval range
stats_a2 = topic_cluster_stats[topic_cluster_stats['topic'] != -1].copy()
avg_cols = avg_cluster_columns(stats_a2)
if len(avg_cols) < 2:
    save_unavailable_figure(
        'figure_A2a_disagreement_direction_bar',
        'Disagreement Direction Unavailable',
        'At least two cluster approval columns are required.'
    )
    save_unavailable_figure(
        'figure_A2b_disagreement_direction_scatter',
        'Disagreement Scatter Unavailable',
        'At least two cluster approval columns are required.'
    )
else:
    values = stats_a2[avg_cols].astype(float)
    stats_a2['approval_min'] = values.min(axis=1)
    stats_a2['approval_max'] = values.max(axis=1)
    stats_a2['approval_range'] = stats_a2['approval_max'] - stats_a2['approval_min']
    stats_a2['abs_gap'] = stats_a2['approval_range']
    stats_a2['top_cluster'] = values.idxmax(axis=1).map(cluster_id_from_avg_col)
    stats_a2['bottom_cluster'] = values.idxmin(axis=1).map(cluster_id_from_avg_col)

    top50_a2 = stats_a2.nlargest(50, 'abs_gap').sort_values('abs_gap', ascending=True)
    colors_a2 = [cluster_color.get(int(cid), color_for_cluster(int(cid))) for cid in top50_a2['top_cluster']]

    fig, ax = plt.subplots(figsize=(10, 16))
    ax.barh(top50_a2[TOPIC_LABEL_COL], top50_a2['abs_gap'], color=colors_a2, alpha=0.85, edgecolor='white', linewidth=0.4)
    legend_handles_a2 = [
        Line2D([0], [0], color=cluster_color.get(int(cid), color_for_cluster(int(cid))), lw=6, label=f"{cluster_role_name.get(int(cid), f'Cluster {int(cid)}')} highest")
        for cid in sorted(stats_a2['top_cluster'].dropna().unique())
    ]
    ax.legend(handles=legend_handles_a2, frameon=False, loc='lower right', fontsize=8.5)
    ax.set_title('Top 50 Topics by Cross-Cluster Approval Range', fontsize=12)
    ax.set_xlabel('Approval range across clusters (max − min)')
    ax.set_ylabel('Topic')
    ax.tick_params(axis='y', labelsize=8.5)
    fig.tight_layout()
    save_figure(fig, 'figure_A2a_disagreement_direction_bar')

    from adjustText import adjust_text

    fig, ax = plt.subplots(figsize=(8, 7))
    sc_a2 = ax.scatter(
        stats_a2['approval_min'],
        stats_a2['approval_max'],
        c=stats_a2['top_cluster'],
        cmap='tab10',
        s=stats_a2['notes'] * 4,
        alpha=0.65,
        edgecolors='white', linewidths=0.4,
    )
    ax.plot([0, 1], [0, 1], '--', color=PALETTE['slate'], linewidth=1.0, alpha=0.7, label='No disagreement')

    labeled_rows = stats_a2.nlargest(15, 'abs_gap')
    texts = []
    for _, row in labeled_rows.iterrows():
        t = ax.text(row['approval_min'] + 0.01, row['approval_max'] + 0.01, row[TOPIC_LABEL_COL][:32], fontsize=8, alpha=0.92, color=PALETTE['ink'], ha='left')
        texts.append(t)

    adjust_text(
        texts,
        x=labeled_rows['approval_min'].values,
        y=labeled_rows['approval_max'].values,
        ax=ax,
        expand_text=(1.15, 1.4),
        expand_points=(1.4, 1.4),
        force_text=(0.4, 0.5),
        force_points=(0.3, 0.3),
        lim=500,
        arrowprops=dict(arrowstyle='-', color=PALETTE['slate'], lw=0.5, alpha=0.6),
    )

    plt.colorbar(sc_a2, ax=ax, label='Cluster with highest mean approval')
    ax.set_title('Per-Topic Cross-Cluster Approval Range\n(dot size proportional to note count)', fontsize=12)
    ax.set_xlabel('Lowest cluster mean approval')
    ax.set_ylabel('Highest cluster mean approval')
    ax.legend(frameon=False, loc='upper left', fontsize=8.5)
    fig.tight_layout()
    save_figure(fig, 'figure_A2b_disagreement_direction_scatter')


## Figure A3: Cluster-Level User Profiling

In [ ]:
# Figure A3 — Cluster-Level User Profiling
user_stats_a3 = (
    ratings_clustered
    .groupby(['raterParticipantId', 'cluster'])
    .agg(total_votes=('vote', 'count'), positive_votes=('vote', 'sum'))
    .reset_index()
)
user_stats_a3['positive_rate'] = user_stats_a3['positive_votes'] / user_stats_a3['total_votes']
cluster_ids = sorted(user_stats_a3['cluster'].dropna().astype(int).unique())

fig, axes = plt.subplots(1, 2, figsize=(13.8, 5.6))

for cluster_id in cluster_ids:
    data = user_stats_a3[user_stats_a3['cluster'] == cluster_id]['total_votes']
    axes[0].hist(data, bins=60, alpha=0.48, color=cluster_color.get(cluster_id, color_for_cluster(cluster_id)), label=cluster_role_name.get(cluster_id, f'Cluster {cluster_id}'), density=True)
axes[0].set_xscale('log')
axes[0].set_title('Vote Count Distribution by Voting Bloc\n(log scale)', fontsize=11)
axes[0].set_xlabel('Total votes per user (log scale)')
axes[0].set_ylabel('Density')
axes[0].legend(frameon=False)

box_data = [user_stats_a3[user_stats_a3['cluster'] == cluster_id]['positive_rate'].values for cluster_id in cluster_ids]
bp = axes[1].boxplot(
    box_data,
    tick_labels=[cluster_role_name.get(cluster_id, f'Cluster {cluster_id}') for cluster_id in cluster_ids],
    patch_artist=True,
    medianprops=dict(color=PALETTE['ink'], linewidth=2.0),
    boxprops=dict(linewidth=0.8),
    whiskerprops=dict(linewidth=0.8),
    capprops=dict(linewidth=0.8),
    flierprops=dict(marker='o', markersize=2.5, alpha=0.3),
)
for patch, cluster_id in zip(bp['boxes'], cluster_ids):
    patch.set_facecolor(cluster_color.get(cluster_id, color_for_cluster(cluster_id)))
    patch.set_alpha(0.75)
axes[1].set_title('Positive Vote Rate Distribution by Voting Bloc', fontsize=11)
axes[1].set_ylabel('Positive vote rate')
axes[1].set_ylim(0, 1)
axes[1].axhline(0.5, color=PALETTE['slate'], linestyle='--', linewidth=0.9, alpha=0.6)
axes[1].tick_params(axis='x', rotation=15)

fig.suptitle('Voting-Bloc Behavioral Profiles: Activity and Approval Tendencies', fontsize=13, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'figure_A3_user_profiling')


## Saved Figure Targets

Expected outputs after running this notebook:
- `figures/paper/figure_01_cluster_diagnostics.pdf`
- `figures/paper/figure_03_strategy_status_counts.pdf`
- `figures/paper/figure_04_high_disagreement_topics.pdf`
- `figures/paper/figure_05_topic_cluster_polarity.pdf`
- `figures/paper/figure_06_gabriel_politics.pdf`
- `figures/paper/figure_07_gabriel_ratings.pdf`
- `figures/paper/figure_08_troll_topic_pockets.pdf`
- `figures/paper/figure_A1_cluster_tfidf.pdf`
- `figures/paper/figure_A2a_disagreement_direction_bar.pdf`
- `figures/paper/figure_A2b_disagreement_direction_scatter.pdf`
- `figures/paper/figure_A3_user_profiling.pdf`
